In [35]:
import os
import pandas as pd
import numpy as np
import joblib

In [37]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

In [39]:
# making the model and pipeline files for future uses
MODEL_FILE = "model.pkl"
PIPELINE_FILE = "pipeline.pkl"

In [43]:
 
def build_pipeline(num_attribs, cat_attribs):
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", cat_pipeline, cat_attribs)
    ])
    return full_pipeline

In [48]:
if not os.path.exists(MODEL_FILE):
    # TRAINING PHASE
    housing = pd.read_csv("housing.csv")
    housing['income_cat'] = pd.cut(housing["median_income"], 
                                   bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf], 
                                   labels=[1, 2, 3, 4, 5])
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    for train_index, test_index in split.split(housing, housing['income_cat']):
        inference_input = housing.loc[test_index].drop(["income_cat", "median_house_value"], axis=1)
        inference_input.to_csv("input.csv", index=False)
        housing = housing.loc[train_index].drop("income_cat", axis=1)
        
 
    housing_labels = housing["median_house_value"].copy()
    housing_features = housing.drop("median_house_value", axis=1)
 
    num_attribs = housing_features.drop("ocean_proximity", axis=1).columns.tolist()
    cat_attribs = ["ocean_proximity"]
 
    pipeline = build_pipeline(num_attribs, cat_attribs)
    housing_prepared = pipeline.fit_transform(housing_features)
 
    model = RandomForestRegressor(random_state=42)
    model.fit(housing_prepared, housing_labels)
 
    # Save model and pipeline
    joblib.dump(model, MODEL_FILE)
    joblib.dump(pipeline, PIPELINE_FILE)
    print("Model trained and saved.")

else:
    # INFERENCE PHASE
    print("Loading serialized production artifacts...")
    model = joblib.load(MODEL_FILE)
    pipeline = joblib.load(PIPELINE_FILE)
 
    # Load un-labeled data
    input_data = pd.read_csv("input.csv")
    
    # Transform via identical training configurations
    transformed_input = pipeline.transform(input_data)
    predictions = model.predict(transformed_input)
 
    # Create a clean export DataFrame to preserve the raw input file
    output_df = input_data.copy()
    output_df["predicted_median_house_value"] = predictions
 
    # Save results to a separate file
    output_df.to_csv("output.csv", index=False)
    print("Inference complete. Results saved cleanly to output.csv")

Inference complete. Results saved to output.csv
